# Train-Ready Dataset Build

This notebook builds the kinase activity manifest from the raw KLIFS and ChEMBL inputs, then splits and saves train-ready artifacts under `data/processed/`.


In [ ]:
import importlib.util
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from torch.utils.data import DataLoader, Dataset

ROOT = Path("/ocean/projects/cis260039p/mjaju/projects")
sys.path.insert(0, str(ROOT.parent))

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)


In [ ]:
script_path = ROOT / "scripts" / "build_kinase_manifest.py"
spec = importlib.util.spec_from_file_location("build_kinase_manifest", script_path)
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)

builder.main()

manifest = pd.read_csv(builder.OUTPUT_MANIFEST)
print("processed manifest shape:", manifest.shape)
print(manifest.head())


In [ ]:
required_cols = [
    "complex_id",
    "pdb_id",
    "protein_chain",
    "ligand_resname",
    "ligand_smiles",
    "target_name",
    "species",
    "uniprot_id",
    "label_type",
    "label_value",
    "label_unit",
    "activity_type",
    "n_measurements",
    "molecule_chembl_ids",
    "template_structure_id",
    "template_quality_score",
    "template_resolution",
    "template_ligand_resname",
    "data_source",
    "pdb_path",
]
missing_cols = [col for col in required_cols if col not in manifest.columns]
print("missing columns:", missing_cols)

manifest = manifest.copy()
manifest["label_value"] = pd.to_numeric(manifest["label_value"], errors="coerce")
manifest = manifest.dropna(subset=["ligand_smiles", "pdb_path", "label_value"]).reset_index(drop=True)
manifest["scaffold"] = manifest["ligand_smiles"].map(
    lambda smiles: Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(Chem.MolFromSmiles(smiles)))
    if Chem.MolFromSmiles(smiles) is not None
    else ""
)
manifest["target_key"] = manifest["target_name"].astype(str) + " | " + manifest["species"].astype(str)
print("rows after cleanup:", len(manifest))
print("duplicate rows:", manifest.duplicated(subset=["ligand_smiles", "pdb_path", "label_value"]).sum())
print(manifest[["complex_id", "target_name", "species", "label_value", "pdb_path"]].head())


In [ ]:
def scaffold_split(frame, frac_train=0.8, frac_valid=0.1, frac_test=0.1, seed=42):
    rng = random.Random(seed)
    scaffold_groups = {}
    for idx, scaffold in enumerate(frame["scaffold"].fillna("")):
        scaffold_groups.setdefault(scaffold, []).append(idx)

    groups = list(scaffold_groups.values())
    rng.shuffle(groups)
    groups = sorted(groups, key=len, reverse=True)

    train_limit = int(len(frame) * frac_train)
    valid_limit = int(len(frame) * frac_valid)
    train_ids, valid_ids, test_ids = [], [], []

    for group in groups:
        if len(train_ids) + len(group) <= train_limit:
            train_ids.extend(group)
        elif len(valid_ids) + len(group) <= valid_limit:
            valid_ids.extend(group)
        else:
            test_ids.extend(group)

    if len(test_ids) == 0 and len(valid_ids) > 0:
        test_ids = valid_ids[-1:]
        valid_ids = valid_ids[:-1]

    return frame.iloc[train_ids].copy(), frame.iloc[valid_ids].copy(), frame.iloc[test_ids].copy()


train_df, valid_df, test_df = scaffold_split(manifest)
print("split sizes:", len(train_df), len(valid_df), len(test_df))
print("train target count:", train_df["target_key"].nunique())
print("valid target count:", valid_df["target_key"].nunique())
print("test target count:", test_df["target_key"].nunique())

class KinaseManifestDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        return (
            row["ligand_smiles"],
            row["pdb_path"],
            torch.tensor(float(row["label_value"]), dtype=torch.float32),
        )


def manifest_collate(batch):
    smiles, pdb_paths, labels = zip(*batch)
    return list(smiles), list(pdb_paths), torch.stack(labels)

train_dataset = KinaseManifestDataset(train_df)
valid_dataset = KinaseManifestDataset(valid_df)
test_dataset = KinaseManifestDataset(test_df)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=manifest_collate)
preview_batch = next(iter(train_loader))
print("preview smiles:", preview_batch[0][:2])
print("preview pdb paths:", preview_batch[1][:2])
print("preview labels:", preview_batch[2][:2])


In [ ]:
try:
    from projects.dataset import LigandPocketDataset, collate_fn as graph_collate_fn

    graph_records = list(train_df[["ligand_smiles", "pdb_path", "label_value"]].itertuples(index=False, name=None))[:2]
    graph_dataset = LigandPocketDataset(graph_records, use_3d=False)
    graph_loader = DataLoader(graph_dataset, batch_size=2, shuffle=False, collate_fn=graph_collate_fn)
    graph_batch = next(iter(graph_loader))
    print("graph batch ligand graphs:", graph_batch[0].num_graphs)
    print("graph batch labels shape:", graph_batch[3].shape)
except Exception as exc:
    print("graph dataset validation skipped:", repr(exc))


In [ ]:
processed_dir = ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

full_manifest_path = processed_dir / "kinase_activity_manifest.csv"
train_path = processed_dir / "kinase_activity_train.csv"
valid_path = processed_dir / "kinase_activity_valid.csv"
test_path = processed_dir / "kinase_activity_test.csv"
summary_path = processed_dir / "kinase_activity_manifest_summary.txt"
metadata_path = processed_dir / "kinase_activity_manifest_metadata.json"

manifest_to_save = manifest.drop(columns=["scaffold"])
train_to_save = train_df.drop(columns=["scaffold"])
valid_to_save = valid_df.drop(columns=["scaffold"])
test_to_save = test_df.drop(columns=["scaffold"])

manifest_to_save.to_csv(full_manifest_path, index=False)
train_to_save.to_csv(train_path, index=False)
valid_to_save.to_csv(valid_path, index=False)
test_to_save.to_csv(test_path, index=False)

summary_text = "\n".join(
    [
        f"Total rows: {len(manifest_to_save)}",
        f"Train rows: {len(train_to_save)}",
        f"Validation rows: {len(valid_to_save)}",
        f"Test rows: {len(test_to_save)}",
        f"Unique targets: {manifest_to_save['target_key'].nunique()}",
        f"Unique scaffolds: {manifest['scaffold'].nunique()}",
    ]
) + "\n"
summary_path.write_text(summary_text)

metadata = {
    "full_manifest": str(full_manifest_path),
    "train_manifest": str(train_path),
    "valid_manifest": str(valid_path),
    "test_manifest": str(test_path),
    "row_counts": {
        "total": len(manifest_to_save),
        "train": len(train_to_save),
        "valid": len(valid_to_save),
        "test": len(test_to_save),
    },
    "unique_targets": int(manifest_to_save["target_key"].nunique()),
    "unique_scaffolds": int(manifest["scaffold"].nunique()),
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

print("wrote:", full_manifest_path)
print("wrote:", train_path)
print("wrote:", valid_path)
print("wrote:", test_path)
print("wrote:", summary_path)
print("wrote:", metadata_path)
print(summary_text)
